# Distance Score according to Weinan et al. (2025)

- D = |A_near − A_far|/max(A_near, A_far)
- Pearson correlation

to compute differences between near and far trials over sessions for: 
- All neurons
- Top n neurons of ensemble
- Ensmeble

In [2]:
import os
import sys
import re
import numpy as np
import scipy.stats as stats
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.join(os.getcwd(), '..', '..', '..'))
from baseVR.base_functionality import init_import_paths
init_import_paths() 

from CustomLogger import CustomLogger as Logger
from analytics_processing import analytics

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args, fullfnames2snames
from dashsrc.plot_components.plots import plot_TrackFiringRate
from dashsrc.plot_components.plots import plot_unit_fr_stability

In [3]:
Logger().init_logger(None, None, logging_level="DEBUG")
animal_ids = [6]
paradigm = [1100]
session_range = [1,33]
session_ids = None
normalize = True
smooth = False
excl_session_names = None  #['2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min', '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min', '2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min']

session_dirs = sessionlist_fullfnames_from_args(paradigm, animal_ids, session_ids, excl_session_names=excl_session_names)[0]
session_names= fullfnames2snames(session_dirs)

2026-02-23 18:06:39,253|DEBUG|71238|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
2026-02-23 18:06:40,275|DEBUG|71238|sessions_from_nas_parsing|get_sessionlist_fullfnames
	For paradigms [1100], animals [6], found 34 sessions.
2026-02-23 18:06:40,276|DEBUG|71238|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: [1100], animal_ids: [6], session_ids: None, from_date: None, to_date: None
	Merging 34 sessions



In [4]:
# firing rates and behavior data
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
fr_track = analytics.get_analytics('FiringRateTrackwiseHz', session_names=session_names)
# fr_z_scored  = analytics.get_analytics('FiringRate40msZ', session_names=session_names)
# fr_z_all_sess = fr.apply(lambda unit_fr: ((unit_fr - unit_fr.mean()) / unit_fr.std()))
#behav = analytics.get_analytics('BehaviorTrackwise', session_names=session_names)
#behav.index = behav.index.droplevel(('animal_id', 'paradigm_id', 'entry_id'))

2026-02-23 18:06:40,292|DEBUG|71238|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-23 18:06:40,565|DEBUG|71238|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 34 sessions

2026-02-23 18:06:40,567|DEBUG|71238|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-23 18:06:40,578|INFO|71238|analytics|get_analytics
	Analytic `FiringRate40msHz` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-23 18:06:40,579|DEBUG|71238|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min.hdf5

In [5]:
# distance scoire function following sun et al.
def get_distance_score(A_near, A_far):
    denom = np.maximum(A_near, A_far)
    out = np.zeros_like(denom, dtype=float)
    m = denom != 0
    out[m] = np.abs(A_near[m] - A_far[m]) / denom[m]
    return out

distance_scores_cue = []
distance_scores_r1 = []
distance_scores_r2  = []

for session_id in fr_track.index.get_level_values('session_id').unique():
    drop_session_data = fr_track.xs(session_id, level='session_id')
    
    a_near_cue = []
    a_near_r1 = []
    a_near_r2 = []

    a_far_cue = []
    a_far_r1 = []
    a_far_r2 = []

    # max for each trial and append to near or far depending on the cue
    for trial_id in drop_session_data['trial_id'].values.unique():
        trial_data = drop_session_data[drop_session_data['trial_id'] == trial_id]  # ['Assembly012']
        mask_cols = trial_data.columns.str.startswith("Unit")
        a_trial_cue = trial_data.loc[trial_data['from_position_bin'].between(-80, 25), mask_cols].max().max()
        a_trial_r1 = trial_data.loc[trial_data['from_position_bin'].between(50, 110), mask_cols].max().max()
        a_trial_r2 = trial_data.loc[trial_data['from_position_bin'].between(170, 230), mask_cols].max().max()
        if (trial_data['cue'] == 1).all():
            a_near_cue.append(a_trial_cue)
            a_near_r1.append(a_trial_r1)
            a_near_r2.append(a_trial_r2)
        elif (trial_data['cue'] == 2).all():
            a_far_cue.append(a_trial_cue)
            a_far_r1.append(a_trial_r1)
            a_far_r2.append(a_trial_r2)
        else:
            print(f"Warning: trial {trial_id} in session {session_id} has mixed cues. Skipping.")
    distance_score_cue = get_distance_score(np.mean(a_near_cue), np.mean(a_far_cue))
    distance_score_r1 = get_distance_score(np.mean(a_near_r1), np.mean(a_far_r1))
    distance_score_r2 = get_distance_score(np.mean(a_near_r2), np.mean(a_far_r2))
    

    distance_scores_cue.append({'session_id': session_id, 'distance_score': distance_score_cue})
    distance_scores_r1.append({'session_id': session_id, 'distance_score': distance_score_r1})
    distance_scores_r2.append({'session_id': session_id, 'distance_score': distance_score_r2})


distance_scores_cue
    
        # distance_score = get_distance_score(trial_data)
        # fr_track.loc[(slice(None), trial_id), 'distance_score'] = distance_score

[{'session_id': '2024-11-14_16-40', 'distance_score': array(0.00963489)},
 {'session_id': '2024-11-15_15-48', 'distance_score': array(0.01491094)},
 {'session_id': '2024-11-20_17-46', 'distance_score': array(0.05494505)},
 {'session_id': '2024-11-21_17-22', 'distance_score': array(0.02857143)},
 {'session_id': '2024-11-25_16-25', 'distance_score': array(0.00246305)},
 {'session_id': '2024-11-26_16-39', 'distance_score': array(0.0253035)},
 {'session_id': '2024-11-28_17-41', 'distance_score': array(0.05655406)},
 {'session_id': '2024-11-29_17-21', 'distance_score': array(0.24)},
 {'session_id': '2024-12-02_16-09', 'distance_score': array(0.0675257)},
 {'session_id': '2024-12-03_16-23', 'distance_score': array(0.11969062)},
 {'session_id': '2024-12-04_18-06', 'distance_score': array(0.02737542)},
 {'session_id': '2024-12-06_16-49', 'distance_score': array(0.06759129)},
 {'session_id': '2024-12-09_17-45', 'distance_score': array(0.07523148)},
 {'session_id': '2024-12-10_17-20', 'distance_

In [6]:
def get_distance_score(A_near, A_far):
    denom = np.maximum(A_near, A_far)
    out = np.zeros_like(denom, dtype=float)
    m = denom != 0
    out[m] = np.abs(A_near[m] - A_far[m]) / denom[m]
    return out

windows = {
    "cue": (-80, 25),
    "r1":  (50, 110),
    "r2":  (170, 230),
}

unit_prefix = "Unit"
results = []

for session_id in fr_track.index.get_level_values("session_id").unique():
    drop_session_data = fr_track.xs(session_id, level="session_id")
    unit_cols = drop_session_data.columns[drop_session_data.columns.str.startswith(unit_prefix)]

    near = {k: [] for k in windows}
    far  = {k: [] for k in windows}

    for trial_id in drop_session_data["trial_id"].unique():
        trial_data = drop_session_data[drop_session_data["trial_id"] == trial_id]

        if (trial_data["cue"] == 1).all():
            group = near
        elif (trial_data["cue"] == 2).all():
            group = far
        else:
            continue

        for wname, (lo, hi) in windows.items():
            s = (
                trial_data
                .loc[trial_data["from_position_bin"].between(lo, hi), unit_cols]
                .mean(axis=0)   # mean activation per unit
            )
            group[wname].append(s)

    for wname in windows:
        n1 = len(near[wname])
        n2 = len(far[wname])
        n_pair = min(n1, n2)

        #  distance scores according to Sun et al. 
        if n1 == 0 or n2 == 0:
            dist = pd.Series(np.nan, index=unit_cols)
        else:
            A_near = pd.concat(near[wname], axis=1).mean(axis=1)
            A_far  = pd.concat(far[wname],  axis=1).mean(axis=1)
            dist = pd.Series(
                get_distance_score(A_near.to_numpy(), A_far.to_numpy()),
                index=unit_cols
            )

        # Pearson correlaation
        if n_pair < 2:
            pearson_r = pd.Series(np.nan, index=unit_cols)
            pearson_p = pd.Series(np.nan, index=unit_cols)
        else:
            X = pd.concat(near[wname][:n_pair], axis=1)
            Y = pd.concat(far[wname][:n_pair],  axis=1)

            r_vals = np.empty(len(unit_cols))
            p_vals = np.empty(len(unit_cols))

            for i, u in enumerate(unit_cols):
                x = X.loc[u].to_numpy()
                y = Y.loc[u].to_numpy()

                if np.allclose(x, x[0]) or np.allclose(y, y[0]):
                    r_vals[i] = np.nan
                    p_vals[i] = np.nan
                else:
                    r_vals[i], p_vals[i] = stats.pearsonr(x, y)

            pearson_r = pd.Series(r_vals, index=unit_cols)
            pearson_p = pd.Series(p_vals, index=unit_cols)

        tmp = pd.DataFrame({
            "unit": unit_cols,
            "distance_score": dist.values,
            "pearson_r": pearson_r.values,
            "pearson_p": pearson_p.values,
            "session_id": session_id,
            "window": wname,
            "n_trials_cue1": n1,
            "n_trials_cue2": n2,
            "n_pairs_used": n_pair,
        })

        results.append(tmp)

distance_metrics_per_unit = (
    pd.concat(results, ignore_index=True)
      .set_index(["session_id", "window", "unit"])
      .sort_index()
)

distance_metrics_per_unit


distance_score  pearson_r  pearson_p  \
session_id       window unit                                             
2024-11-14_16-40 cue    Unit0001        0.124314   0.124783   0.398088   
                        Unit0002        0.545154   0.001538   0.991723   
                        Unit0003        0.336319  -0.180217   0.220298   
                        Unit0004        1.000000        NaN        NaN   
                        Unit0005        0.173686   0.100867   0.495147   
...                                          ...        ...        ...   
2025-01-27_13-39 r2     Unit0073        0.164680   0.058281   0.621844   
                        Unit0074        0.018181   0.034119   0.772898   
                        Unit0075        0.039534   0.121352   0.303033   
                        Unit0076        0.079092   0.197802   0.091157   
                        Unit0077        0.018635   0.187120   0.110400   

                                  n_trials_cue1  n_trials_cue2  n_pairs_used  
session_id       window unit                                                  
2024-11-14_16-40 cue    Unit0001             54             48            48  
                        Unit0002             54             48            48  
                        Unit0003             54             48            48  
                        Unit0004             54             48            48  
                        Unit0005             54             48            48  
...                                         ...            ...           ...  
2025-01-27_13-39 r2     Unit0073             74             78            74  
                        Unit0074             74             78            74  
                        Unit0075             74             78            74  
                        Unit0076             74             78            74  
                        Unit0077             74             78            74  

[6699 rows x 6 columns]

In [7]:
df = distance_metrics_per_unit.reset_index()

# Ensure consistent row ordering
window_order = ["cue", "r1", "r2"]
df = df[df["window"].isin(window_order)].copy()
df["window"] = pd.Categorical(df["window"], categories=window_order, ordered=True)

# Pick up to 10 sessions
session_ids = (
    df["session_id"]
    .drop_duplicates()
    .tolist()
)[:30]

n_rows = 3
n_cols = 30

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    row_titles=window_order,
    column_titles=[str(s) for s in session_ids] + [""] * (n_cols - len(session_ids)),
    horizontal_spacing=0.02,
    vertical_spacing=0.06,
)

x_range = [-1.0, 1.0]   
y_range = [0.0, 1.0] 

for c, session_id in enumerate(session_ids, start=1):
    d_sess = df[df["session_id"] == session_id]

    for r, w in enumerate(window_order, start=1):
        d = d_sess[d_sess["window"] == w]

        # Filter invalid points
        d = d[np.isfinite(d["pearson_r"]) & np.isfinite(d["distance_score"])]

        fig.add_trace(
            go.Scatter(
                x=d["pearson_r"],
                y=d["distance_score"],
                mode="markers",
                marker=dict(size=5, opacity=0.7),
                hovertemplate=(
                    "session=%{customdata[0]}<br>"
                    "window=%{customdata[1]}<br>"
                    "unit=%{customdata[2]}<br>"
                    "r=%{x:.3f}<br>"
                    "dist=%{y:.3f}<extra></extra>"
                ),
                customdata=np.stack(
                    [d["session_id"].astype(str), d["window"].astype(str), d["unit"].astype(str)],
                    axis=1
                ),
                showlegend=False,
            ),
            row=r,
            col=c,
        )

        # Per-subplot axes formatting
        fig.update_xaxes(range=x_range, row=r, col=c, zeroline=True)
        fig.update_yaxes(range=y_range, row=r, col=c, zeroline=True)

# Axis labels: only on outer plots to reduce clutter
for c in range(1, len(session_ids) + 1):
    fig.update_xaxes(title_text="Pearson r", row=3, col=c)
for r in range(1, 4):
    fig.update_yaxes(title_text="Distance score", row=r, col=1)

fig.update_layout(
    height=800,
    width=14400,
    margin=dict(l=5, r=5, t=80, b=40),
    title_text="Per-unit distance score vs cue1–cue2 Pearson correlation (trial-wise mean activation)",
)

fig.show()
